# MLflow Model Registry - Cervical Cancer Classifier

## Purpose and Benefits of the MLflow Model Registry

The **MLflow Model Registry** is a centralized catalog that manages the full lifecycle of machine learning models. It solves several critical MLOps challenges:

### Key Benefits

| Benefit | Description |
|---------|-------------|
| **Centralized Model Catalog** | All trained models stored in one place with metadata, metrics, and artifacts |
| **Model Versioning** | Every registration creates a new version (v1, v2, v3...). Roll back if needed |
| **Lifecycle Stage Management** | Models move through stages: None, Staging, Production, Archived |
| **Audit Trail** | Every transition is logged with timestamps |
| **Consistent Loading** | Load via models:/Name/Production URI - no path hardcoding |
| **Collaboration** | Teams share models and promote candidates together

---
## Section 1: Setup

Configure MLflow to use SQLite as the backend store.

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from mlflow.tracking import MlflowClient

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
REGISTRY_DB = PROJECT_ROOT / 'notebooks' / 'mlflow.db'
mlflow.set_tracking_uri(f"sqlite:///{REGISTRY_DB.as_posix()}")
mlflow.set_experiment("Cervical_Cancer_Registry")

print(f"MLflow version: {mlflow.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Registry database: {REGISTRY_DB}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print("Registry will use the leakage-safe pipelines produced by model training.")

---
## Section 2: Load Preprocessed Data

Load data prepared in the feature engineering step.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'processed_data'
X_train = np.load(OUTPUT_DIR / 'X_train.npy')
X_val = np.load(OUTPUT_DIR / 'X_val.npy')
X_test = np.load(OUTPUT_DIR / 'X_test.npy')
y_train = np.load(OUTPUT_DIR / 'y_train.npy')
y_val = np.load(OUTPUT_DIR / 'y_val.npy')
y_test = np.load(OUTPUT_DIR / 'y_test.npy')

with open(OUTPUT_DIR / 'feature_columns.json', 'r') as f:
    feature_columns = json.load(f)

print(
    f"Data loaded: X_train {X_train.shape}, X_val {X_val.shape}, "
    f"X_test {X_test.shape}, features={len(feature_columns)}"
)

---
## Section 3: Train Multiple Models with MLflow Tracking

Each model logs parameters, metrics, and the model artifact.

In [ ]:
models = {}
predictions = {}
probabilities = {}
metrics_dict = {}
run_ids = {}

model_files = {
    'Logistic_Regression': 'Logistic_Regression_tuned.pkl',
    'Logistic_ElasticNet': 'Logistic_ElasticNet_tuned.pkl',
    'Linear_SVM': 'Linear_SVM_tuned.pkl',
    'RBF_SVM': 'RBF_SVM_tuned.pkl',
    'Random_Forest': 'Random_Forest_tuned.pkl',
    'Gradient_Boosting': 'Gradient_Boosting_tuned.pkl',
    'LDA_Shrinkage': 'LDA_Shrinkage_tuned.pkl',
    'SGD_ElasticNet': 'SGD_ElasticNet_tuned.pkl',
    'Hist_Gradient_Boosting': 'Hist_Gradient_Boosting_tuned.pkl',
}

print("Registering leakage-safe trained pipelines with MLflow...")

for model_name, filename in model_files.items():
    model_path = PROJECT_ROOT / 'models' / filename
    if not model_path.exists():
        print(f"Skipping {model_name}: {model_path} not found")
        continue

    model = joblib.load(model_path)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    models[model_name] = model
    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    metrics_dict[model_name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'PR-AUC': average_precision_score(y_test, y_prob),
    }

    with mlflow.start_run(run_name=f"Registry_{model_name}") as run:
        run_ids[model_name] = run.info.run_id
        mlflow.log_param('model_type', model_name)
        mlflow.log_param('source_artifact', filename)
        mlflow.log_param('preprocessing', 'fold-local imputer, scaler, and SMOTE')
        mlflow.log_param('decision_threshold', 0.5)
        for metric_name, metric_value in metrics_dict[model_name].items():
            mlflow.log_metric(metric_name.lower().replace('-', '_'), round(metric_value, 4))
        mlflow.sklearn.log_model(model, 'model', serialization_format='pickle')

    print(
        f"  {model_name:24s} | "
        f"F1: {metrics_dict[model_name]['F1-Score']:.4f} | "
        f"PR-AUC: {metrics_dict[model_name]['PR-AUC']:.4f}"
    )

if not models:
    raise FileNotFoundError("No trained model artifacts were found in models/.")
print(f"All {len(models)} available leakage-safe pipelines tracked successfully.")

---
## Section 4: Select the Production Model

Test-set F1 is shown for reference only. Selection uses validation F1 among models with a train-validation F1 gap of at most 0.05 (the same rule as the training and evaluation notebooks).

In [ ]:
# Test-set metrics above are logged for reference only. The Production model is
# chosen with the same rule as the training/evaluation notebooks: highest
# validation F1 among models whose train-validation F1 gap is <= 0.05
# (ties broken by validation PR-AUC).
MAX_TRAIN_VAL_GAP = 0.05
results_df = pd.DataFrame(metrics_dict).T.reset_index().rename(columns={'index': 'Model'}).round(4)

training_summary = pd.read_csv(PROJECT_ROOT / 'models' / 'model_comparison_advanced.csv')
training_summary = training_summary[training_summary['Model'].isin(metrics_dict)]
candidates = training_summary[training_summary['train_val_gap'] <= MAX_TRAIN_VAL_GAP]
if candidates.empty:
    candidates = training_summary
selection_ranking = candidates.sort_values(['val_f1_score', 'val_pr_auc'], ascending=False)['Model'].tolist()
best_model_name = selection_ranking[0]

print("Test-set metrics (reference only, not used for selection):")
print(results_df.sort_values('F1-Score', ascending=False).to_string(index=False))
print(f"\nSelected for Production: {best_model_name}")
print(training_summary.set_index('Model').loc[best_model_name, ['val_f1_score', 'train_val_gap']].to_string())
print(f"Run ID: {run_ids[best_model_name]}")


---
## Section 5: Register All Models

Register all 9 trained models so they can be used via the API.
Each model gets its own name in the registry.

In [ ]:
print("Registering all models...")
print("=" * 70)

registered_models = {}

for model_name, run_id in run_ids.items():
    # Use model name directly as the registered model name
    model_uri = f"runs:/{run_id}/model"
    
    try:
        model_version = mlflow.register_model(model_uri=model_uri, name=model_name)
        registered_models[model_name] = {
            'version': model_version.version,
            'run_id': run_id,
            'f1_score': metrics_dict[model_name]['F1-Score']
        }
        print(f"✓ Registered: {model_name:25s} | v{model_version.version} | F1: {metrics_dict[model_name]['F1-Score']:.4f}")
    except Exception as exc:
        print(f"✗ Failed to register {model_name}: {exc}")

print("=" * 70)
print(f"\nTotal models registered: {len(registered_models)}")
print(f"\nRegistered models:")
for name, info in sorted(registered_models.items(), key=lambda x: x[1]['f1_score'], reverse=True):
    print(f"  {name:25s} | v{info['version']} | F1: {info['f1_score']:.4f}")

---
## Section 6: Model Versioning

Each call to register_model() under the same name creates a new version.
Register v1, then v2, then v3 - each with its own metrics and stage.

In [ ]:
print("Model versions created:")
print("=" * 70)

client = MlflowClient()
for model_name in registered_models.keys():
    versions = client.search_model_versions(f"name='{model_name}'")
    if versions:
        print(f"\n{model_name}:")
        for v in sorted(versions, key=lambda x: int(x.version)):
            print(f"  v{v.version}: stage={v.current_stage}, status={v.status}")

---
## Section 7: Set Best Model to Production

Transition the model selected in Section 4 to the Production stage.
All other models remain in None stage.

In [ ]:
best_model_info = registered_models[best_model_name]

print(f"Setting best model to Production...")
print(f"Best Model: {best_model_name} (test F1: {best_model_info['f1_score']:.4f})")

# Transition best model to Production
client.transition_model_version_stage(
    name=best_model_name,
    version=best_model_info['version'],
    stage="Production"
)
print(f"✓ {best_model_name} v{best_model_info['version']} is now in Production")

print(f"\nCurrent model stages:")
for model_name in sorted(registered_models.keys()):
    info = registered_models[model_name]
    versions = client.search_model_versions(f"name='{model_name}'")
    if versions:
        stage = versions[0].current_stage
        stage_symbol = "✓" if model_name == best_model_name else "-"
        print(f"  {stage_symbol} {model_name:25s} | v{info['version']} | {stage}")

---
## Section 8: View in MLflow UI

Run in terminal:
```bash
cd notebooks
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Open http://localhost:5000 -> Click 'Models' tab

You will see:
- 8 registered models (one for each algorithm)
- Best model in Production stage
- Metrics, parameters, run IDs for each model
- Stage badges and transition buttons

---
## Section 9: Load Model from Registry for Inference

Use the models:/ URI scheme:
- models:/Gradient_Boosting/Production
- models:/Linear_SVM/None
- models:/Gradient_Boosting/None

No file paths needed - MLflow resolves the version by stage.

In [ ]:
print("Testing model loading from registry...")
print("=" * 70)

# Load the best model (Production)
production_model_uri = f"models:/{best_model_name}/Production"
production_model = mlflow.pyfunc.load_model(model_uri=production_model_uri)
print(f"✓ Loaded Production model: {best_model_name}")

# Test prediction
X_test_df = pd.DataFrame(X_test, columns=feature_columns)
y_pred_production = production_model.predict(X_test_df)
acc = accuracy_score(y_test, y_pred_production)
f1 = f1_score(y_test, y_pred_production)
print(f"  Accuracy:  {acc:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Load another model (e.g., second best)
if len(selection_ranking) > 1:
    second_model_name = selection_ranking[1]
    second_model_uri = f"models:/{second_model_name}/None"
    try:
        second_model = mlflow.pyfunc.load_model(model_uri=second_model_uri)
        y_pred_second = second_model.predict(X_test_df)
        acc2 = accuracy_score(y_test, y_pred_second)
        f1_2 = f1_score(y_test, y_pred_second)
        print(f"\n✓ Loaded second model: {second_model_name}")
        print(f"  Accuracy:  {acc2:.4f}")
        print(f"  F1-Score:  {f1_2:.4f}")
    except Exception as exc:
        print(f"\n✗ Could not load {second_model_name}: {exc}")

print("\n" + "=" * 70)
print("Model registry setup complete!")
print("=" * 70)
print(f"\nYou can now use these models in the API:")
for model_name in sorted(registered_models.keys()):
    stage = "Production" if model_name == best_model_name else "None"
    print(f"  - {model_name:25s} (stage: {stage})")

---
## Section 10: Programmatic Registry Management with MlflowClient

Useful for automation, dashboards, and enforcing governance rules.

In [ ]:
print("All registered models:")
for rm in client.search_registered_models():
    print(f"  - {rm.name}")
print()

# Update descriptions for all registered models
print("Updating model descriptions...")
for model_name, info in registered_models.items():
    f1_score = info['f1_score']
    stage = "Production" if model_name == best_model_name else "None"
    description = f"Cervical Cancer Risk Classifier ({model_name}). Test Test F1: {f1_score:.4f}. Stage: {stage}."
    client.update_registered_model(name=model_name, description=description)
    client.update_model_version(
        name=model_name,
        version=info['version'],
        description=f"{model_name} model. Test F1: {f1_score:.4f}. Stage: {stage}."
    )
print(f"✓ Updated descriptions for {len(registered_models)} models")

print("\nQuery models by stage:")
for stage in ["None", "Staging", "Production", "Archived"]:
    models_in_stage = []
    for model_name in registered_models.keys():
        mv = client.get_latest_versions(model_name, stages=[stage])
        if mv:
            models_in_stage.append(model_name)
    if models_in_stage:
        print(f"  {stage}: {', '.join(models_in_stage)}")
    else:
        print(f"  {stage}: (none)")

print("\n" + "=" * 70)
print("MLflow Model Registry setup complete!")
print("=" * 70)
print("\nKey takeaways:")
print(f"  1. All {len(registered_models)} models registered with their names")
print("  2. Best model is in Production stage")
print("  3. Load via models:/ModelName/Stage URIs")
print("  4. Use in API: ?model=Random_Forest or ?model=Linear_SVM (default: Production model)")

---
## Summary: MLOps Best Practices

### Do's
- Register every candidate model (complete history)
- Use descriptive version descriptions
- Always validate in Staging before promoting to Production
- Load by stage, not version (code auto-picks up promotions)
- Archive deprecated models

### Don'ts
- Hardcode file paths (use models:/Name/Production instead)
- Skip staging (bypasses validation)
- Delete old versions (archive, you may need rollback)
- Ignore descriptions (versions become opaque numbers)

### View Results
```bash
cd notebooks
mlflow ui --backend-store-uri sqlite:///mlflow.db

# Open http://localhost:5000 -> Models tab
```

```bash
cd notebooks
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```
